## OSCAR Baseline Notebook Part B — ResNet18 training and evaluation. 

Standalone training notebook. Reads synthetic images from Part 1 notebook output and real test images from dataset. This notebook does not use blip/clip models since dataset generation was already done on client side in previous notebooks.

In [ ]:
!pip install -q pillow tqdm

In [2]:
import os
import json
import random
import shutil
import warnings
import datetime
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 28
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Torch:", torch.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  {props.total_memory // 1024**2} MB")


Device: cuda
Torch: 2.10.0+cu128
GPU: Tesla T4  14912 MB


In [ ]:
@dataclass
class Config:
    dataset_name: str = "domainnet"

    part1_root: str = "/kaggle/input/datasets/aliazhar21/domainnet-409topk"

    data_root: str = "/kaggle/input/datasets/mei1963/domainnet/DomainNet"

    # Must match Part 1 configurations here
    max_classes: int = 12
    images_per_class_test: int = 100

    # Top-K copmression, used only in outputs and json files here. Compression was applied before SD generation in last part. 
    topk_k: int = 64

    batch_size: int = 32
    epochs: int = 50
    lr: float = 1e-4
    weight_decay: float = 1e-2
    dropout_p: float = 0.2
    grad_clip: float = 1.0
    patience: int = 10

    work_dir: str = "/kaggle/working/oscar_part2_domainnet"
    outputs_dir: str = "/kaggle/working/outputs"

CFG = Config()

WORK = Path(CFG.work_dir)
for sub in ["checkpoints", "metrics", "plots"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)
OUTPUTS = Path(CFG.outputs_dir)
OUTPUTS.mkdir(parents=True, exist_ok=True)
(OUTPUTS / "plots").mkdir(parents=True, exist_ok=True)



SYNTH_DIR = Path(CFG.part1_root) / "oscar_baseline_domainnet" / "synthetic"
print("Checking:", SYNTH_DIR)
print("Exists:", SYNTH_DIR.exists())
print("Parent exists:", SYNTH_DIR.parent.exists())
print("Parent contents:", list(SYNTH_DIR.parent.iterdir()) if SYNTH_DIR.parent.exists() else "parent missing")
print("part1_root contents:", list(Path(CFG.part1_root).iterdir()) if Path(CFG.part1_root).exists() else "part1_root missing")


assert SYNTH_DIR.exists(), f"Synthetic dir not found: {SYNTH_DIR}"
print("Synthetic dir:", SYNTH_DIR)
print("Config ready.")


Checking: /kaggle/input/datasets/aliazhar21/domainnet-409topk/oscar_baseline_domainnet/synthetic
Exists: True
Parent exists: True
Parent contents: [PosixPath('/kaggle/input/datasets/aliazhar21/domainnet-409topk/oscar_baseline_domainnet/synthetic'), PosixPath('/kaggle/input/datasets/aliazhar21/domainnet-409topk/oscar_baseline_domainnet/captions'), PosixPath('/kaggle/input/datasets/aliazhar21/domainnet-409topk/oscar_baseline_domainnet/embeddings')]
part1_root contents: [PosixPath('/kaggle/input/datasets/aliazhar21/domainnet-409topk/outputs'), PosixPath('/kaggle/input/datasets/aliazhar21/domainnet-409topk/oscar_baseline_domainnet'), PosixPath('/kaggle/input/datasets/aliazhar21/domainnet-409topk/__huggingface_repos__.json')]
Synthetic dir: /kaggle/input/datasets/aliazhar21/domainnet-409topk/oscar_baseline_domainnet/synthetic
Config ready.


In [4]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def is_image_file(p: Path) -> bool:
    return p.suffix.lower() in IMG_EXTS

def load_image(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")

class LabeledImageDataset(Dataset):
    def __init__(self, records, class_to_idx, transform=None):
        self.records = records
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, cls, _split = self.records[idx]
        image = load_image(Path(path))
        if self.transform is not None:
            image = self.transform(image)
        return image, self.class_to_idx[cls]


In [5]:
# ── Load synthetic training images from Part 1 output ────────────────────────
train_source = "synthetic"
synth_records = []
classes_found = []
for cls_dir in sorted(SYNTH_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    imgs = [p for p in cls_dir.iterdir() if is_image_file(p)]
    if not imgs:
        print(f"  WARNING: no images for {cls_dir.name}")
        continue
    classes_found.append(cls_dir.name)
    for p in imgs:
        synth_records.append((p, cls_dir.name, "train"))

if not synth_records:
    raise RuntimeError(f"No synthetic images found at {SYNTH_DIR}")

class_to_idx_used = {cls: i for i, cls in enumerate(sorted(classes_found))}
print(f"Synthetic: {len(classes_found)} classes, {len(synth_records)} images")
print(f"Classes: {list(class_to_idx_used.keys())}")

# ── Build real test records from DomainNet official test split ────────────────
def load_domainnet_test_records(root: Path, classes_found):
    cls_set = set(classes_found)
    records = []
    for txt in sorted(root.glob("*_test.txt")):
        with open(txt) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rel_path = line.split()[0]
                parts = Path(rel_path).parts
                if len(parts) < 3:
                    continue
                cls = parts[1]
                if cls not in cls_set:
                    continue
                abs_path = root / rel_path
                if abs_path.exists():
                    records.append((abs_path, cls, "test"))
    return records

domainnet_root = Path(CFG.data_root)
rng = random.Random(SEED)
all_test = load_domainnet_test_records(domainnet_root, classes_found)

# Sample up to images_per_class_test per class, consistent with Part A
by_cls = defaultdict(list)
for rec in all_test:
    by_cls[rec[1]].append(rec)
test_records = []
for cls in sorted(classes_found):
    pool = sorted(by_cls[cls], key=lambda r: str(r[0]))
    rng.shuffle(pool)
    test_records.extend(pool[:CFG.images_per_class_test])

print(f"Test records: {len(test_records)} from DomainNet")


Synthetic: 12 classes, 2400 images
Classes: ['beard', 'bird', 'dog', 'golf_club', 'shoe', 'spreadsheet', 'squirrel', 'table', 'tiger', 'tree', 'whale', 'windmill']
Test records: 1200 from DomainNet


In [6]:
train_tfm = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    T.RandomRotation(15),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
test_tfm = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = LabeledImageDataset(synth_records, class_to_idx_used, transform=train_tfm)
test_ds = LabeledImageDataset(test_records, class_to_idx_used, transform=test_tfm)

train_loader = DataLoader(
    train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train: {len(train_ds)} images  |  {len(train_loader)} batches/epoch")
print(f"Test:  {len(test_ds)} images")


Train: 2400 images  |  75 batches/epoch
Test:  1200 images


In [7]:
# Fresh ResNet-18 load — no accelerate in this kernel so no meta tensor risk
base_model = resnet18(weights=ResNet18_Weights.DEFAULT)
in_features = base_model.fc.in_features
base_model.fc = nn.Sequential(
    nn.Dropout(p=CFG.dropout_p),
    nn.Linear(in_features, len(class_to_idx_used)),
)
model = base_model.to(DEVICE)
print(f"Model on {DEVICE} | {len(class_to_idx_used)} classes")

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs, eta_min=1e-6)

best_acc = 0.0
best_path = WORK / "checkpoints" / f"{CFG.dataset_name}_resnet18_best.pt"
history = []
epochs_no_improve = 0

def run_eval(model, loader):
    model.eval()
    total, correct, losses = 0, 0, []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            losses.append(criterion(logits, y).item())
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return (float(np.mean(losses)) if losses else 0.0), (correct / max(total, 1))

print(f"Training: {CFG.epochs} epochs, patience={CFG.patience}")
for epoch in range(CFG.epochs):
    model.train()
    running = []
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        optimizer.step()
        running.append(loss.item())
    scheduler.step()

    train_loss = float(np.mean(running)) if running else 0.0
    val_loss, val_acc = run_eval(model, test_loader)
    current_lr = scheduler.get_last_lr()[0]
    history.append({"epoch": epoch+1, "train_loss": train_loss,
                     "test_loss": val_loss, "test_acc": val_acc, "lr": current_lr})

    if val_acc > best_acc or (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}: train={train_loss:.4f}  test={val_loss:.4f}  acc={val_acc:.4f}  lr={current_lr:.2e}")

    if val_acc > best_acc:
        best_acc = val_acc
        epochs_no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_to_idx": class_to_idx_used,
            "config": CFG.__dict__,
        }, best_path)
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= CFG.patience:
        print(f"Early stopping at epoch {epoch+1}.")
        break

checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Best test accuracy: {best_acc:.4f}")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


Model on cuda | 12 classes
Training: 50 epochs, patience=10
Epoch   1: train=2.0360  test=2.0173  acc=0.4417  lr=9.99e-05
Epoch   2: train=1.5679  test=1.8931  acc=0.4950  lr=9.96e-05
Epoch   5: train=1.1964  test=2.2064  acc=0.4800  lr=9.76e-05
Epoch   6: train=1.1215  test=2.0392  acc=0.5008  lr=9.65e-05
Epoch  10: train=0.9258  test=2.0801  acc=0.4708  lr=9.05e-05
Epoch  15: train=0.8006  test=2.0528  acc=0.4792  lr=7.96e-05
Early stopping at epoch 16.
Best test accuracy: 0.5008


In [8]:
topk_tag = f"topk{CFG.topk_k}" if CFG.topk_k is not None else "full"
run_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Copy checkpoint to outputs
resnet_out = OUTPUTS / f"{CFG.dataset_name}_resnet18_{topk_tag}_{run_ts}.pt"
shutil.copy(best_path, resnet_out)
print(f"Checkpoint -> {resnet_out}")

# Training curve plot
plot_name = f"{CFG.dataset_name}_{topk_tag}_{run_ts}.png"
plot_path = OUTPUTS / "plots" / plot_name
_ep = [h["epoch"] for h in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(_ep, [h["train_loss"] for h in history], label="Train loss")
ax1.plot(_ep, [h["test_loss"] for h in history], label="Test loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss curves")
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(_ep, [h["test_acc"] for h in history], color="green", label="Test acc")
ax2.axhline(best_acc, color="red", linestyle="--", label=f"Best={best_acc:.4f}")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.set_title(f"Test accuracy [{topk_tag}]")
ax2.legend(); ax2.grid(True, alpha=0.3)
fig.suptitle(
    f"OSCAR ResNet-18 | {CFG.dataset_name} | {topk_tag}",
    fontsize=10,
)
fig.tight_layout()
fig.savefig(str(plot_path), dpi=120, bbox_inches="tight")
plt.close(fig)
print(f"Plot -> {plot_path}")

# Persistent runs log — appends across multiple topk runs
runs_log_path = OUTPUTS / "runs_log.json"
run_entry = {
    "timestamp": run_ts,
    "dataset": CFG.dataset_name,
    "topk_k": CFG.topk_k,
    "max_classes": CFG.max_classes,
    "epochs_run": len(history),
    "best_test_acc": round(best_acc, 6),
    "lr": CFG.lr,
    "batch_size": CFG.batch_size,
    "dropout_p": CFG.dropout_p,
    "weight_decay": CFG.weight_decay,
    "train_source": train_source,
    "plot_file": plot_name,
    "checkpoint_file": str(resnet_out),
    "epoch_history": history,
}
_existing = []
if runs_log_path.exists():
    try:
        with open(runs_log_path) as _f:
            _existing = json.load(_f)
    except Exception:
        _existing = []
_existing.append(run_entry)
with open(runs_log_path, "w") as _f:
    json.dump(_existing, _f, indent=2)
print(f"Run #{len(_existing)} logged -> {runs_log_path}")

# Summary metrics
metrics_path = WORK / "metrics" / f"{CFG.dataset_name}_{topk_tag}_summary.json"
history_path = WORK / "metrics" / f"{CFG.dataset_name}_{topk_tag}_history.csv"
with open(metrics_path, "w") as f:
    json.dump({"dataset": CFG.dataset_name, "topk_k": CFG.topk_k,
               "best_test_accuracy": best_acc, "history": history}, f, indent=2)
pd.DataFrame(history).to_csv(history_path, index=False)
print("Metrics saved:", metrics_path)


Checkpoint -> /kaggle/working/outputs/domainnet_resnet18_topk64_20260507_213013.pt
Plot -> /kaggle/working/outputs/plots/domainnet_topk64_20260507_213013.png
Run #1 logged -> /kaggle/working/outputs/runs_log.json
Metrics saved: /kaggle/working/oscar_part2_domainnet/metrics/domainnet_topk64_summary.json
